# 03 — Defining a common sample for all representations

Meaningful model comparison requires every representation to be evaluated on the same locations. This notebook combines the corrected 20,000-location EPC sample with 6,597 PTAL locations, preserves their existing IDs and coordinates, and establishes a common sample of 26,597 observations.

It also measures whether the selected PTAL and EPC samples differ materially from their larger cleaned populations, verifies the dimensions and sample coverage of each representation, and separates Street View visual content from information about Street View availability.

## Summary of the output

The common sample contains 20,000 EPC and 6,597 PTAL observations across all 33 London boroughs. Two EPC rows had inconsistent free-text borough labels; both are assigned to the correct official borough code so that spatial validation uses a single, stable geography.

SatCLIP, TESSERA, AlphaEarth and DINOv2 cover 100% of the common sample. Street View covers 98.02% of EPC locations and 86.90% of PTAL locations. The missing Street View rows reflect the absence of a suitable panorama within the search neighbourhood, not a failed table join. Differences between the modelling samples and the larger cleaned populations are small, so the established sample is retained.

In [ ]:
# Connect Google Drive and load packages used for sample and coverage summaries.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FINAL_CODE_DIR = Path('/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE')
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({
    name: getattr(_config, name)
    for name in dir(_config)
    if not name.startswith('_')
})

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 200)

required_paths = {
    'corrected EPC 20k': EPC_20K_FINAL_CANDIDATE,
    'legacy PTAL clean': LEGACY_PTAL_CLEAN,
    'common DINO sample': DINO_SAMPLE_PATH,
    'DINOv2': DINO2_EMB_PATH,
    'TESSERA': TESSERA_EMB_PATH,
    'SatCLIP': SATCLIP_EMB_PATH,
    'AlphaEarth final': ALPHA_FINAL_PATH,
    'Street View': STREET_EMB_PATH,
}

for name, path in required_paths.items():
    print(f'{name:20s} | {path.exists()} | {path}')

## 1. Construct the common sample

PTAL rows are taken from the established representation sample, while EPC values are replaced with the corrected results from Notebook 01. Sample IDs and coordinates remain unchanged. Borough geography is represented by official ONS London-borough codes rather than free-text names, preventing minor spelling or naming differences from changing the spatial train/test divisions.

In [ ]:
# Combine the corrected EPC branch with the established PTAL sample and borough codes.
dino_sample = pd.read_csv(DINO_SAMPLE_PATH)
epc20k = pd.read_parquet(EPC_20K_FINAL_CANDIDATE)

dino_sample["sample_id"] = dino_sample["sample_id"].astype(str)
epc20k["sample_id"] = epc20k["sample_id"].astype(str)

ptal_common = dino_sample[
    dino_sample["task"].astype(str).str.upper() == "PTAL"
].copy()

legacy_epc_ids = set(
    dino_sample.loc[
        dino_sample["task"].astype(str).str.upper() == "EPC",
        "sample_id"
    ].astype(str)
)
corrected_epc_ids = set(epc20k["sample_id"].astype(str))

assert len(ptal_common) == 6597, f"Expected 6,597 PTAL rows, found {len(ptal_common)}"
assert len(epc20k) == 20000, f"Expected 20,000 EPC rows, found {len(epc20k)}"
assert legacy_epc_ids == corrected_epc_ids, (
    "Corrected EPC sample IDs do not match existing embedding sample IDs."
)
assert epc20k["label_regression"].notna().all(), (
    "Corrected EPC target contains missing values."
)

# ------------------------------------------------------------------
# Canonical borough geography for later spatial cross-validation.
# ------------------------------------------------------------------
# EPC already contains an ONS London-borough local-authority code.
# Use that code as the authoritative grouping key and derive the display name
# from the canonical E09 mapping rather than trusting a potentially dirty name.
epc20k["borough_original"] = epc20k["borough"].astype("string")
epc20k["borough_code"] = epc20k["local_authority"].astype("string")
epc20k["borough"] = epc20k["borough_code"].map(LONDON_BOROUGH_CODE_TO_NAME)

unmapped_epc_codes = sorted(
    epc20k.loc[epc20k["borough"].isna(), "borough_code"].dropna().unique().tolist()
)
assert not unmapped_epc_codes, (
    f"Unrecognised EPC London-borough codes: {unmapped_epc_codes}"
)

epc20k["borough_label_corrected"] = (
    epc20k["borough_original"].fillna("").str.strip()
    != epc20k["borough"].astype("string").fillna("").str.strip()
)

borough_label_audit = epc20k.loc[
    epc20k["borough_label_corrected"],
    ["sample_id", "postcode_clean", "local_authority",
     "borough_original", "borough"]
].copy()

print("EPC borough labels corrected from canonical ONS code:",
      len(borough_label_audit))
if len(borough_label_audit):
    display(borough_label_audit)

borough_label_audit.to_csv(
    AUDIT_DIR / "epc20k_borough_canonicalisation_audit.csv",
    index=False
)

# PTAL has borough names but no local-authority code in the legacy common table.
# Reverse-map the canonical names to create the same stable grouping key.
ptal_common["borough"] = ptal_common["borough"].astype("string").str.strip()
ptal_common["borough_code"] = ptal_common["borough"].map(
    LONDON_BOROUGH_NAME_TO_CODE
)

unmapped_ptal_names = sorted(
    ptal_common.loc[
        ptal_common["borough_code"].isna(), "borough"
    ].dropna().unique().tolist()
)
assert not unmapped_ptal_names, (
    f"Unrecognised PTAL borough names: {unmapped_ptal_names}"
)

common_base = pd.concat(
    [ptal_common, epc20k],
    ignore_index=True,
    sort=False
)

assert len(common_base) == 26597
assert common_base["sample_id"].is_unique
assert common_base["label_regression"].notna().all()
assert common_base["borough_code"].notna().all()
assert common_base["borough"].notna().all()
assert common_base["borough_code"].nunique() == 33

print("Final common sample:", common_base.shape)
display(common_base["task"].value_counts().rename("n").to_frame())
print("Canonical borough groups:", common_base["borough_code"].nunique())

common_base.to_parquet(COMMON_SAMPLE_FINAL_PATH, index=False)
print("Saved:", COMMON_SAMPLE_FINAL_PATH)

## 2. Compare the samples with the larger cleaned populations

The PTAL sample was spatially thinned on a 500-metre grid, while the EPC sample was stratified by borough and EPC level. This section compares each selected sample with the corresponding larger population.

The standardised mean difference describes any shift in the target distribution, and total variation distance describes any shift in borough shares. Both are scale-free descriptive measures rather than hypothesis tests. Joint borough-by-EPC-level proportions are also examined for the EPC sample.

In [ ]:
# Compare selected samples with their larger cleaned populations.
ptal_full = pd.read_csv(LEGACY_PTAL_CLEAN)
epc_full_path = AUDIT_DIR / "epc_postcode_corrected_candidate_v2.parquet"
epc_full = pd.read_parquet(epc_full_path)

# Canonicalise borough geography without heuristic majority voting.
ptal_full["borough"] = ptal_full["borough"].astype("string").str.strip()
ptal_full["borough_code"] = ptal_full["borough"].map(
    LONDON_BOROUGH_NAME_TO_CODE
)
assert ptal_full["borough_code"].notna().all(), (
    "Some full-clean PTAL rows could not be mapped to canonical borough codes."
)

epc_full["borough_code"] = epc_full["local_authority"].astype("string")
epc_full["borough"] = epc_full["borough_code"].map(
    LONDON_BOROUGH_CODE_TO_NAME
)

unknown_epc_codes = sorted(
    epc_full.loc[
        epc_full["borough"].isna(), "borough_code"
    ].dropna().unique().tolist()
)
assert not unknown_epc_codes, (
    f"Corrected full EPC contains unrecognised London-borough codes: {unknown_epc_codes}"
)
assert epc_full["borough"].notna().all()
print("Corrected full EPC borough coverage: 100.0000%")
print("Corrected full EPC borough groups:", epc_full["borough_code"].nunique())


def target_summary(df, target_col, name):
    s = pd.to_numeric(df[target_col], errors="coerce").dropna()
    q = s.quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    return {
        "dataset": name,
        "n": int(len(s)),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "q01": float(q.loc[0.01]),
        "q05": float(q.loc[0.05]),
        "q25": float(q.loc[0.25]),
        "median": float(q.loc[0.50]),
        "q75": float(q.loc[0.75]),
        "q95": float(q.loc[0.95]),
        "q99": float(q.loc[0.99]),
    }


representativeness = pd.DataFrame([
    target_summary(ptal_full, "label_regression", "PTAL full clean"),
    target_summary(ptal_common, "label_regression", "PTAL common sample"),
    target_summary(epc_full, "current_energy_efficiency",
                   "EPC corrected full postcode"),
    target_summary(epc20k, "label_regression", "EPC common sample"),
])
display(representativeness)
representativeness.to_csv(SAMPLE_REPRESENTATIVENESS_PATH, index=False)


def borough_share_difference(full_df, sample_df, group_col="borough_code"):
    full = (
        full_df[group_col].astype("string")
        .value_counts(normalize=True)
    )
    sample = (
        sample_df[group_col].astype("string")
        .value_counts(normalize=True)
    )
    idx = full.index.union(sample.index)

    out = pd.DataFrame({
        "full_share": full.reindex(idx, fill_value=0),
        "sample_share": sample.reindex(idx, fill_value=0),
    })
    out["difference_pp"] = (
        out["sample_share"] - out["full_share"]
    ) * 100
    out["abs_difference_pp"] = out["difference_pp"].abs()
    out["borough"] = out.index.map(LONDON_BOROUGH_CODE_TO_NAME)
    return out.sort_values("abs_difference_pp", ascending=False)


def standardized_mean_difference(full_values, sample_values):
    full_values = pd.to_numeric(full_values, errors="coerce").dropna()
    sample_values = pd.to_numeric(sample_values, errors="coerce").dropna()

    pooled_sd = np.sqrt(
        (full_values.var(ddof=1) + sample_values.var(ddof=1)) / 2
    )
    return float(
        (sample_values.mean() - full_values.mean()) / pooled_sd
    )


def total_variation_distance(share_table):
    return float(
        0.5 * (
            share_table["sample_share"]
            - share_table["full_share"]
        ).abs().sum()
    )


ptal_borough_diff = borough_share_difference(
    ptal_full, ptal_common
)
epc_borough_diff = borough_share_difference(
    epc_full, epc20k
)

print("Largest PTAL borough-share differences (percentage points)")
display(
    ptal_borough_diff[
        ["borough", "full_share", "sample_share",
         "difference_pp", "abs_difference_pp"]
    ].head(10)
)

print("Largest EPC borough-share differences (percentage points)")
display(
    epc_borough_diff[
        ["borough", "full_share", "sample_share",
         "difference_pp", "abs_difference_pp"]
    ].head(10)
)

ptal_smd = standardized_mean_difference(
    ptal_full["label_regression"],
    ptal_common["label_regression"]
)
epc_smd = standardized_mean_difference(
    epc_full["current_energy_efficiency"],
    epc20k["label_regression"]
)

ptal_tvd = total_variation_distance(ptal_borough_diff)
epc_tvd = total_variation_distance(epc_borough_diff)

design_effect = pd.DataFrame([
    {
        "task": "PTAL",
        "target_standardized_mean_difference": ptal_smd,
        "borough_total_variation_distance": ptal_tvd,
        "max_abs_borough_share_difference_pp":
            float(ptal_borough_diff["abs_difference_pp"].max()),
        "full_n": int(len(ptal_full)),
        "sample_n": int(len(ptal_common)),
    },
    {
        "task": "EPC",
        "target_standardized_mean_difference": epc_smd,
        "borough_total_variation_distance": epc_tvd,
        "max_abs_borough_share_difference_pp":
            float(epc_borough_diff["abs_difference_pp"].max()),
        "full_n": int(len(epc_full)),
        "sample_n": int(len(epc20k)),
    },
])

# ---------------------------------------------------------------
# EPC joint-stratum audit: borough × EPC level
# ---------------------------------------------------------------
# The original 20k EPC sample was drawn within these strata, so verify the
# joint distribution directly rather than checking boroughs alone.
def epc_rating_to_level(value):
    if pd.isna(value):
        return np.nan
    v = str(value).strip().upper()
    if v in {"A", "B"}:
        return "high"
    if v in {"C", "D"}:
        return "medium"
    if v in {"E", "F", "G"}:
        return "low"
    return np.nan


epc_full["epc_level_audit"] = epc_full[
    "current_energy_rating"
].map(epc_rating_to_level)

sample_level_col = (
    "epc_level"
    if "epc_level" in epc20k.columns
    else "corrected_epc_level"
)
epc20k["epc_level_audit"] = (
    epc20k[sample_level_col]
    .astype("string")
    .str.lower()
)

assert epc_full["epc_level_audit"].notna().all(), (
    "Some corrected full EPC postcodes have no valid EPC level."
)
assert epc20k["epc_level_audit"].notna().all(), (
    "Some EPC common-sample rows have no valid EPC level."
)

epc_full["_stratum"] = (
    epc_full["borough_code"].astype(str)
    + "|"
    + epc_full["epc_level_audit"].astype(str)
)
epc20k["_stratum"] = (
    epc20k["borough_code"].astype(str)
    + "|"
    + epc20k["epc_level_audit"].astype(str)
)

full_strata_share = epc_full["_stratum"].value_counts(normalize=True)
sample_strata_share = epc20k["_stratum"].value_counts(normalize=True)
strata_idx = full_strata_share.index.union(sample_strata_share.index)

epc_strata_diff = pd.DataFrame({
    "full_share": full_strata_share.reindex(strata_idx, fill_value=0),
    "sample_share": sample_strata_share.reindex(strata_idx, fill_value=0),
})
epc_strata_diff["difference_pp"] = (
    epc_strata_diff["sample_share"]
    - epc_strata_diff["full_share"]
) * 100
epc_strata_diff["abs_difference_pp"] = (
    epc_strata_diff["difference_pp"].abs()
)
epc_strata_diff["borough_code"] = [
    x.split("|", 1)[0] for x in epc_strata_diff.index
]
epc_strata_diff["epc_level"] = [
    x.split("|", 1)[1] for x in epc_strata_diff.index
]
epc_strata_diff["borough"] = epc_strata_diff[
    "borough_code"
].map(LONDON_BOROUGH_CODE_TO_NAME)

epc_strata_tvd = float(
    0.5 * (
        epc_strata_diff["sample_share"]
        - epc_strata_diff["full_share"]
    ).abs().sum()
)

print("Largest EPC borough × level stratum differences")
display(
    epc_strata_diff.sort_values(
        "abs_difference_pp", ascending=False
    )[
        ["borough", "epc_level", "full_share", "sample_share",
         "difference_pp", "abs_difference_pp"]
    ].head(15)
)

# Check whether the original min_per_group=5 safeguard could have affected
# proportional allocation for the 20k target.
strata_counts = epc_full["_stratum"].value_counts()
proportional_alloc = (
    20000 * strata_counts / len(epc_full)
).round().astype(int)

n_strata_total = int(len(strata_counts))
n_strata_proportional_below5 = int(
    (proportional_alloc < 5).sum()
)

print("EPC strata:", n_strata_total)
print(
    "Strata whose proportional 20k allocation is <5:",
    n_strata_proportional_below5
)
print("EPC joint-strata TVD:", epc_strata_tvd)

epc_strata_diff.to_csv(
    AUDIT_DIR / "epc_sample_joint_strata_difference.csv"
)

# Add the joint-stratum diagnostics to the EPC design-effect row.
design_effect.loc[
    design_effect["task"] == "EPC",
    "joint_strata_total_variation_distance"
] = epc_strata_tvd

design_effect.loc[
    design_effect["task"] == "EPC",
    "max_abs_joint_stratum_difference_pp"
] = float(epc_strata_diff["abs_difference_pp"].max())

design_effect.loc[
    design_effect["task"] == "EPC",
    "n_joint_strata"
] = n_strata_total

design_effect.loc[
    design_effect["task"] == "EPC",
    "n_strata_proportional_allocation_below5"
] = n_strata_proportional_below5

print("Sampling-effect summary")
display(design_effect)
design_effect.to_csv(SAMPLE_DESIGN_EFFECT_PATH, index=False)

ptal_borough_diff.to_csv(
    AUDIT_DIR / "ptal_sample_borough_share_difference.csv"
)
epc_borough_diff.to_csv(
    AUDIT_DIR / "epc_sample_borough_share_difference.csv"
)

In [ ]:
# Plot full-population and selected-sample target distributions.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(pd.to_numeric(ptal_full['label_regression'], errors='coerce').dropna(), bins=40, density=True, alpha=0.45, label='Full clean PTAL')
ax.hist(pd.to_numeric(ptal_common['label_regression'], errors='coerce').dropna(), bins=40, density=True, alpha=0.45, label='Common sample')
ax.set_title('PTAL target distribution: full vs common sample')
ax.set_xlabel('PTAL regression target')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig(FINAL_FIGURE_DIR / 'qa_ptal_full_vs_sample_distribution.png', dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(pd.to_numeric(epc_full['current_energy_efficiency'], errors='coerce').dropna(), bins=40, density=True, alpha=0.45, label='Corrected full EPC postcode')
ax.hist(pd.to_numeric(epc20k['label_regression'], errors='coerce').dropna(), bins=40, density=True, alpha=0.45, label='Common sample')
ax.set_title('EPC target distribution: full vs common sample')
ax.set_xlabel('Mean EPC energy-efficiency score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig(FINAL_FIGURE_DIR / 'qa_epc_full_vs_sample_distribution.png', dpi=180)
plt.show()

## 3. Verify representation coverage and dimensions

Each representation table is matched against the exact 26,597 sample IDs. The summary records duplicate IDs, sample coverage, the number of detected feature columns and the number of rows with a complete feature vector. The expected dimensions are 256 for SatCLIP, 128 for TESSERA, 64 for AlphaEarth, 768 for DINOv2 and 512 for the Street View CLIP representation.

In [ ]:
# Measure ID coverage, duplicates and dimensions for each representation.
representation_specs = {
    "SatCLIP": {
        "path": SATCLIP_EMB_PATH,
        "prefixes": ["satclip_"],
        "expected_dim": 256,
    },
    "TESSERA": {
        "path": TESSERA_EMB_PATH,
        "prefixes": ["tessera_"],
        "expected_dim": 128,
    },
    "AlphaEarth": {
        "path": ALPHA_FINAL_PATH,
        "prefixes": ["alphaearth_2024_"],
        "expected_dim": 64,
    },
    "DINOv2 aerial": {
        "path": DINO2_EMB_PATH,
        "prefixes": ["dinov2_", "dino_"],
        "expected_dim": 768,
    },
    "StreetView CLIP": {
        "path": STREET_EMB_PATH,
        # The original Street View extraction notebook writes the
        # sample-level aggregated features as streetclip_agg_000 ... _511.
        "prefixes": ["streetclip_agg_"],
        "expected_dim": 512,
    },
}

inventory_rows = []
loaded_representations = {}


def detect_feature_cols(df, prefixes):
    for prefix in prefixes:
        cols = [
            c for c in df.columns
            if str(c).startswith(prefix)
        ]
        if cols:
            return cols, prefix
    return [], None


for name, spec in representation_specs.items():
    path = Path(spec["path"])
    if not path.exists():
        raise FileNotFoundError(
            f"{name}: representation file not found: {path}"
        )

    emb = pd.read_parquet(path)
    if "sample_id" not in emb.columns:
        raise KeyError(f"{name}: sample_id column not found.")

    emb["sample_id"] = emb["sample_id"].astype(str)
    duplicate_ids = int(emb["sample_id"].duplicated().sum())
    assert duplicate_ids == 0, (
        f"{name}: found {duplicate_ids} duplicate sample IDs."
    )

    feature_cols, prefix = detect_feature_cols(
        emb, spec["prefixes"]
    )
    assert len(feature_cols) == spec["expected_dim"], (
        f"{name}: detected {len(feature_cols)} features "
        f"with prefix {prefix!r}; expected {spec['expected_dim']}."
    )

    id_coverage = (
        common_base["sample_id"].isin(set(emb["sample_id"])).mean()
        * 100
    )
    assert np.isclose(id_coverage, 100.0), (
        f"{name}: common-sample ID coverage is "
        f"{id_coverage:.4f}%, expected 100%."
    )

    matched = common_base[
        ["sample_id", "task"]
    ].merge(
        emb,
        on="sample_id",
        how="left",
        validate="one_to_one"
    )

    complete_pct = (
        matched[feature_cols]
        .notna()
        .all(axis=1)
        .mean()
        * 100
    )

    inventory_rows.append({
        "representation": name,
        "path_exists": True,
        "n_rows": int(len(emb)),
        "duplicate_sample_ids": duplicate_ids,
        "common_id_coverage_pct": float(id_coverage),
        "detected_prefix": prefix,
        "n_detected_features": int(len(feature_cols)),
        "expected_features": int(spec["expected_dim"]),
        "complete_feature_rows_pct": float(complete_pct),
    })

    loaded_representations[name] = {
        "data": emb,
        "feature_cols": feature_cols,
        "prefix": prefix,
    }


inventory = pd.DataFrame(inventory_rows)
display(inventory)
inventory.to_csv(REPRESENTATION_INVENTORY_PATH, index=False)
print("Saved:", REPRESENTATION_INVENTORY_PATH)

# Non-Street-View embeddings should be complete for all 26,597 rows.
non_sv = inventory[
    inventory["representation"] != "StreetView CLIP"
]
assert np.allclose(
    non_sv["complete_feature_rows_pct"].to_numpy(),
    100.0
), "A supposedly complete non-Street-View representation has missing rows."

## 4. Separate Street View content from availability

The Street View source provides a 512-dimensional visual representation and four accompanying fields describing whether imagery exists, how many panoramas contributed, and their distances from the sample location.

These are retained as three distinct feature options: visual content only, availability metadata only, and visual content plus metadata. When no panorama is available, the visual vector remains missing at this stage. Any numerical filling is learned from training data during model fitting, while the availability fields allow the analysis to distinguish useful visual content from the geography of Street View coverage.

In [ ]:
# Separate Street View visual content from imagery-availability metadata.
sv = loaded_representations["StreetView CLIP"]["data"]
sv_feature_cols = loaded_representations[
    "StreetView CLIP"
]["feature_cols"]

sv_metadata_cols = [
    c for c in [
        "sv_has_streetview",
        "sv_n_images",
        "sv_min_dist_m",
        "sv_mean_dist_m",
    ]
    if c in sv.columns
]

assert len(sv_feature_cols) == 512
assert len(sv_metadata_cols) == 4, (
    f"Expected four Street View metadata features, found: "
    f"{sv_metadata_cols}"
)

sv_check = common_base[
    ["sample_id", "task"]
].merge(
    sv[["sample_id"] + sv_feature_cols + sv_metadata_cols],
    on="sample_id",
    how="left",
    validate="one_to_one"
)

clip_complete = sv_check[
    sv_feature_cols
].notna().all(axis=1)

clip_all_missing = sv_check[
    sv_feature_cols
].isna().all(axis=1)

# No partially missing 512-D vectors should exist.
assert (clip_complete | clip_all_missing).all(), (
    "Street View table contains partially missing CLIP vectors."
)

has_sv = (
    pd.to_numeric(
        sv_check["sv_has_streetview"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
    .astype(bool)
)

assert (clip_complete == has_sv).all(), (
    "Street View CLIP availability does not match "
    "sv_has_streetview."
)

coverage = (
    sv_check
    .groupby("task")
    .agg(
        n_samples=("sample_id", "size"),
        n_with_streetview=("sv_has_streetview", "sum"),
    )
)
coverage["coverage_pct"] = (
    coverage["n_with_streetview"]
    / coverage["n_samples"]
    * 100
)

print("Detected Street View CLIP features:", len(sv_feature_cols))
print("Detected Street View metadata:", sv_metadata_cols)
display(coverage)

coverage.to_csv(
    AUDIT_DIR / "streetview_coverage_by_task.csv"
)

# Final modelling definitions:
# 1. CLIP-only: 512 visual features; missing rows remain NaN here and are
#    imputed INSIDE each training fold in the modelling pipeline.
# 2. Metadata-only: four availability/distance features.
# 3. CLIP + metadata: both sets.
#
# Keeping NaNs at this stage avoids leakage from fitting an imputer on the
# entire dataset before cross-validation.
pd.DataFrame({
    "feature_set": [
        "StreetView_CLIP_only",
        "StreetView_metadata_only",
        "StreetView_CLIP_plus_metadata",
    ],
    "n_clip_features": [
        len(sv_feature_cols), 0, len(sv_feature_cols)
    ],
    "n_metadata_features": [
        0, len(sv_metadata_cols), len(sv_metadata_cols)
    ],
}).to_csv(
    AUDIT_DIR / "streetview_feature_set_definition.csv",
    index=False
)

## 5. Document key sampling and extraction choices

Several settings reflect practical design decisions for this London study: 500-metre PTAL thinning, a 20,000-postcode EPC sample, task-specific Street View neighbourhoods, and aerial crop widths of 300 metres for PTAL and 150 metres for EPC. They are reported as study settings rather than universal standards.

Their implications are addressed through the sample comparisons in this notebook and through later sensitivity analyses of validation geography and representation design. The 50-metre AlphaEarth limit has a narrower role: it is only a ceiling for the 12 verified boundary matches, whose observed maximum distance is 14.42 metres.

## 6. Save the common sample and coverage summary

The common sample is saved only after confirming 26,597 unique IDs, 20,000 EPC observations, 6,597 PTAL observations, 33 borough groups, the expected representation dimensions and documented Street View missingness. These outputs establish the population on which all subsequent model comparisons are based.

In [ ]:
# Save the common-sample and representation-coverage summary.
# Final machine-readable audit summary.
sv_inventory_row = inventory.loc[
    inventory["representation"] == "StreetView CLIP"
].iloc[0]

audit_summary = {
    "n_common_sample": int(len(common_base)),
    "n_epc": int((common_base["task"] == "EPC").sum()),
    "n_ptal": int((common_base["task"] == "PTAL").sum()),
    "n_borough_groups": int(common_base["borough_code"].nunique()),
    "n_epc_borough_labels_corrected": int(
        epc20k["borough_label_corrected"].sum()
    ),
    "ptal_target_smd": float(ptal_smd),
    "epc_target_smd": float(epc_smd),
    "ptal_borough_tvd": float(ptal_tvd),
    "epc_borough_tvd": float(epc_tvd),
    "epc_joint_strata_tvd": float(epc_strata_tvd),
    "epc_joint_strata_max_abs_difference_pp": float(epc_strata_diff["abs_difference_pp"].max()),
    "epc_n_strata_proportional_allocation_below5": int(n_strata_proportional_below5),
    "streetview_clip_dim": int(
        sv_inventory_row["n_detected_features"]
    ),
    "streetview_complete_rows_pct": float(
        sv_inventory_row["complete_feature_rows_pct"]
    ),
    "all_representation_id_coverage_100pct": bool(
        np.allclose(
            inventory["common_id_coverage_pct"].to_numpy(),
            100.0
        )
    ),
    "all_representation_dimensions_match": bool(
        (
            inventory["n_detected_features"]
            == inventory["expected_features"]
        ).all()
    ),
}

assert audit_summary["n_common_sample"] == 26597
assert audit_summary["n_epc"] == 20000
assert audit_summary["n_ptal"] == 6597
assert audit_summary["n_borough_groups"] == 33
assert audit_summary["all_representation_id_coverage_100pct"]
assert audit_summary["all_representation_dimensions_match"]
assert audit_summary["streetview_clip_dim"] == 512

with open(COMMON_AUDIT_SUMMARY_PATH, "w") as f:
    json.dump(audit_summary, f, indent=2)

print("03 audit decision gate: PASS")
display(pd.Series(audit_summary, name="value"))
print("Saved:", COMMON_AUDIT_SUMMARY_PATH)